# Hamilton PREP: Concise demo — teaching needle, liquid transfer, plate movement

Single notebook demonstrating:
1. **Teaching needle** — Pick up teaching tip, move above plate A1 at safe height, drop tip.
2. **Liquid handling** — Tip pickup, dual-channel aspirate and dispense.
3. **Plate movement** — CORE gripper: pick plate from deck[0], drop at deck[1].

**Deck layout:** 1× 50 µL NTR tips at deck[3], 1× plate at deck[0] (moved to deck[1]). Visualizer runs after deck creation.

## 1. Imports and config

In [ ]:
import sys
import logging
from asyncio import sleep

from pylabrobot.liquid_handling.backends.hamilton.prep_backend import PrepBackend
from pylabrobot.liquid_handling.backends.hamilton.tcp_backend import HamiltonTCPClient
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.resources import Coordinate
from pylabrobot.resources.hamilton import PrepDeck
from pylabrobot.resources import hamilton_96_tiprack_50uL_NTR, Cor_Axy_96_wellplate_500uL_Ub
from pylabrobot.visualizer import Visualizer

logging.getLogger("pylabrobot").setLevel(logging.INFO)
logging.getLogger("pylabrobot").handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
logging.getLogger("pylabrobot").addHandler(handler)

HOST = "192.168.100.102"
PORT = 2000
SAFE_HEIGHT_MM_ABOVE_WELL = 25

## 2. Deck layout and visualizer

In [ ]:
# PrepDeck: spots 0–7 (column-major). With CORE grippers for plate movement.
deck = PrepDeck(with_core_grippers=True)

tip_rack = deck[3] = hamilton_96_tiprack_50uL_NTR(name="ntr_50", with_tips=True)
plate = deck[0] = Cor_Axy_96_wellplate_500uL_Ub("plate")

visualizer = Visualizer(deck, open_browser=False)
await visualizer.setup()

## 3. Backend and liquid handler setup

In [ ]:
backend = PrepBackend(host=HOST, port=PORT)
lh = LiquidHandler(backend=backend, deck=deck)
await lh.setup(smart=True, force_initialize=False)

In [ ]:
await backend.print_firmware_tree()

In [ ]:
# Deck, waste, and other config data from setup can be accessed here.
config = lh.backend._config
print("Instrument configuration:")
print(f"  Deck bounds: {config.deck_bounds}")
print(f"  Has enclosure: {config.has_enclosure}")
print(f"  Safe speeds enabled: {config.safe_speeds_enabled}")
print(f"  Default traverse height: {config.default_traverse_height}")
print(f"  Number of channels: {config.num_channels}")
print(f"  Has MPH: {config.has_mph}")

print("\nDeck sites:")
for site in config.deck_sites:
    print(site)

print("\nWaste sites:")
for waste in config.waste_sites:
    print(waste)

In [ ]:
# INTROSPECTION: PREP pipette interface: list command signatures via backend.client.introspect()
PIPETTOR_PATH = "MLPrepRoot.PipettorRoot.Pipettor"
pool, reg = await backend.client.introspect(PIPETTOR_PATH)

print(f"Command signatures on {PIPETTOR_PATH} ({len(reg.methods)} methods):\n")
for m in reg.methods:
    print(f"  {m.get_signature_string(reg)}")

## 4. Teaching needle: above plate A1 at safe height

In [ ]:
# Set Logger to debug so we can see params as sent to the backend.
logging.getLogger("pylabrobot").setLevel(logging.DEBUG)

teaching_tip = deck.get_resource("teaching_tip")
await lh.pick_up_tips([teaching_tip])

a1 = plate.get_item("A1")
safe_pos = a1.get_absolute_location("c", "c", "b") + Coordinate(0, 0, SAFE_HEIGHT_MM_ABOVE_WELL)
await lh.backend.move_to_position(safe_pos.x, safe_pos.y, safe_pos.z)
await sleep(3)

await lh.drop_tips([teaching_tip])

## 5. Tip pickup, aspirate, dispense (dual channel)

In [ ]:
await lh.backend.set_deck_light(0,0,0,0)

In [ ]:
tip_spots = tip_rack['A1:B1']
await lh.pick_up_tips(tip_spots)

await lh.aspirate(
    plate["A1:B1"],
    vols=[35, 25],
    liquid_height=[3, 3],
    z_liquid_exit_speed = [25, 25],
)

await lh.dispense(
    plate['A7:B7'],
    vols=[35, 25],
    liquid_height=[3, 3],
    z_liquid_exit_speed = [25, 25],
)

await lh.drop_tips(tip_spots)
#await lh.discard_tips()

## 6. Plate movement with CORE gripper (deck[0] → deck[1])

In [ ]:
await lh.pick_up_resource(plate)
await lh.drop_resource(destination=deck[1], return_gripper=True)

## 7. Teardown

In [ ]:
logging.getLogger("pylabrobot").setLevel(logging.INFO)

await lh.backend.park()
# await lh.backend.disco_mode()
await lh.stop()